# Day 1 Lab — PyTorch Basics & Linear Networks

**Learning goals**
1. Work with tensors in PyTorch (shapes, broadcasting, GPU).
2. Implement linear regression + gradient descent **from scratch** in NumPy.
3. Let **autograd** compute the same gradients for you.
4. Train a multi-input / multi-output linear network with `nn.Module`, `DataLoader` and an optimiser.
5. Take a first look at real EEG data with a seizure (CHB-MIT).

**Colab:** *Runtime → Change runtime type → T4 GPU*. Exercises are marked with `# TODO`.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)
print("PyTorch", torch.__version__, "| device:", device)

ModuleNotFoundError: No module named 'torch'

## Part 1 — Tensors
A tensor is an n-dimensional array (like a NumPy array) that can live on a GPU and remembers how it was computed.

In [ ]:
a = torch.arange(6, dtype=torch.float32)       # shape (6,)
A = a.reshape(2, 3)                             # shape (2, 3)
B = torch.ones(3, 4)
print(A, A.shape)
print("matrix product:", (A @ B).shape)         # (2, 3) @ (3, 4) -> (2, 4)
print("broadcasting:", A + torch.tensor([10.0, 20.0, 30.0]))   # (2, 3) + (3,) -> (2, 3)
print("to numpy and back:", torch.from_numpy(A.numpy()).dtype)
print("on device:", A.to(device).device)

**Exercise 1.1.** A mini-batch of EEG windows has shape `(batch, channels, time) = (32, 18, 512)`.
1. Create a random tensor `eeg` with this shape (`torch.randn`).
2. Compute the mean of every channel over time → `ch_mean` with shape `(32, 18)`.
3. Standardise every window per channel (subtract the mean, divide by the std over time) → `eeg_std`.
   Hint: `keepdim=True` makes broadcasting work.

In [ ]:
# TODO: create eeg, ch_mean, eeg_std
pass  # YOUR CODE HERE
assert ch_mean.shape == (32, 18)
assert torch.allclose(eeg_std.mean(-1), torch.zeros(32, 18), atol=1e-4)
assert torch.allclose(eeg_std.std(-1), torch.ones(32, 18), atol=1e-4)
print("Exercise 1.1 passed")

## Part 2 — Linear regression from scratch (NumPy)
Model: $\hat y = w x + b$. Loss: $\mathcal L = \frac1n\sum_i (\hat y_i - y_i)^2$.

We generate data from a known line plus noise, so we know the right answer ($w=3$, $b=2$).

In [ ]:
n = 200
x = np.random.uniform(-2, 2, n)
y = 3.0 * x + 2.0 + np.random.normal(0, 0.5, n)
plt.scatter(x, y, s=8); plt.xlabel("x"); plt.ylabel("y"); plt.title("Synthetic data"); plt.show()

**Exercise 2.1.** Derive and implement the gradients
$\frac{\partial \mathcal L}{\partial w} = \frac2n \sum_i (\hat y_i - y_i)\,x_i$ and
$\frac{\partial \mathcal L}{\partial b} = \frac2n \sum_i (\hat y_i - y_i)$.
Complete `mse_and_grads(w, b, x, y)` so that it returns `loss, dw, db`.

In [ ]:
def mse_and_grads(w, b, x, y):
    y_hat = w * x + b
    # TODO: compute loss, dw, db
    pass  # YOUR CODE HERE
    return loss, dw, db

# numerical check with finite differences: never trust a hand-written gradient without it
eps = 1e-6
_, dw, db = mse_and_grads(0.5, -1.0, x, y)
num_dw = (mse_and_grads(0.5 + eps, -1.0, x, y)[0] - mse_and_grads(0.5 - eps, -1.0, x, y)[0]) / (2 * eps)
num_db = (mse_and_grads(0.5, -1.0 + eps, x, y)[0] - mse_and_grads(0.5, -1.0 - eps, x, y)[0]) / (2 * eps)
assert abs(dw - num_dw) < 1e-4 and abs(db - num_db) < 1e-4
print(f"analytic dw={dw:.4f}  numeric dw={num_dw:.4f}")

**Exercise 2.2.** Write the gradient-descent loop: start at `w = b = 0`, run `steps` updates
`w -= lr * dw`, `b -= lr * db` and store the loss of every step in `losses`.

In [ ]:
def gradient_descent(x, y, lr=0.1, steps=100):
    w, b, losses = 0.0, 0.0, []
    # TODO: the update loop
    pass  # YOUR CODE HERE
    return w, b, losses

w, b, losses = gradient_descent(x, y)
print(f"w = {w:.3f}, b = {b:.3f}  (true: 3, 2)")
assert abs(w - 3) < 0.2 and abs(b - 2) < 0.2

**Exercise 2.3 — experiment.** Run gradient descent with learning rates `0.001, 0.05, 0.5, 0.8` and plot the
loss curves (log-scale y-axis). Which one is too slow, which one diverges? Why does it diverge?
(Hint: the largest curvature of this loss is $\approx 2\,\overline{x^2}$; GD diverges if `lr` > 2 / curvature.)

In [ ]:
pass  # YOUR CODE HERE

*Your answer:*

## Part 3 — Autograd
PyTorch records every operation on tensors with `requires_grad=True` and computes gradients with `.backward()`.

In [ ]:
xt, yt = torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
w_t = torch.tensor(0.5, requires_grad=True)
b_t = torch.tensor(-1.0, requires_grad=True)
loss = ((w_t * xt + b_t - yt) ** 2).mean()
loss.backward()
print("autograd:", w_t.grad.item(), b_t.grad.item())
print("by hand: ", *mse_and_grads(0.5, -1.0, x, y)[1:])

**Exercise 3.1.** Re-write the gradient-descent loop with autograd (no hand-written gradients).
Remember: (1) update inside `torch.no_grad()`, (2) reset the gradients with `.grad.zero_()` — otherwise they accumulate!

In [ ]:
w_t = torch.zeros((), requires_grad=True)
b_t = torch.zeros((), requires_grad=True)
lr = 0.1
for step in range(100):
    # TODO: forward pass, backward pass, parameter update, zero the gradients
    pass  # YOUR CODE HERE
print(f"w = {w_t.item():.3f}, b = {b_t.item():.3f}")
assert abs(w_t.item() - w) < 1e-3

## Part 4 — A multi-input / multi-output linear network in PyTorch
California housing: 8 input features per district; we predict **two** targets at once
(median house value and — just to have a 2nd output — the average number of rooms, which we remove from the inputs).
Model: $\hat{\mathbf y} = \mathbf W \mathbf x + \mathbf b$ with $\mathbf W \in \mathbb R^{2\times 7}$ — that is `nn.Linear(7, 2)`.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data = fetch_california_housing()
X_all = np.delete(data.data, 2, axis=1)                          # drop AveRooms from the inputs ...
Y_all = np.stack([data.target, data.data[:, 2]], axis=1)          # ... and predict it as 2nd output
# real data has outliers (e.g. a district with 1243 persons per household); per-sample SGD explodes on them
z = lambda a: np.abs((a - a.mean(0)) / a.std(0))
keep = (z(X_all) < 5).all(1) & (z(Y_all) < 5).all(1)
X_all, Y_all = X_all[keep], Y_all[keep]
print(f"removed {(~keep).sum()} extreme districts")
X_tr, X_te, Y_tr, Y_te = train_test_split(X_all, Y_all, test_size=0.2, random_state=0)
print("inputs", X_tr.shape, "targets", Y_tr.shape, "| feature names:", [f for i, f in enumerate(data.feature_names) if i != 2])

**Exercise 4.1.** Standardise inputs and targets with mean and std computed **on the training set only**
(why only there?). Store the results as float32 tensors `Xtr, Ytr, Xte, Yte`.

In [ ]:
# TODO: standardise with training statistics, convert to float32 tensors
pass  # YOUR CODE HERE
assert Xtr.dtype == torch.float32 and abs(Xtr.mean().item()) < 1e-3

The standard PyTorch training loop — you will write this (with small changes) in every lab of this course.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def train_linear(batch_size=64, lr=0.01, epochs=10, optimiser="sgd"):
    torch.manual_seed(0)
    model = nn.Linear(7, 2).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr) if optimiser == "sgd" else torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    history = []
    for epoch in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = nn.functional.mse_loss(model(xb), yb)   # forward
            opt.zero_grad()                                 # reset gradients
            loss.backward()                                 # backward
            opt.step()                                      # update
        with torch.no_grad():
            history.append(nn.functional.mse_loss(model(Xte.to(device)), Yte.to(device)).item())
    return model, history

model, hist = train_linear()
print("test MSE per epoch:", np.round(hist, 3))
print("learned W shape:", tuple(model.weight.shape), "b shape:", tuple(model.bias.shape))

**Exercise 4.2 — experiment.** Mini-batch size matters. Train with batch sizes `1, 32, 512, len(Xtr)` (full batch)
for the same number of **epochs** and plot the test MSE curves. Also print the wall-clock time per run
(`time.perf_counter()`). Which batch size gives the best result *per epoch*, which *per second*?

In [ ]:
import time
pass  # YOUR CODE HERE

*Your answer:*

**Exercise 4.3.** Compare the trained network with the closed-form least-squares solution
(`torch.linalg.lstsq`, add a column of ones for the bias). How close are the test MSEs? When would you prefer
gradient descent over the closed form?

In [ ]:
pass  # YOUR CODE HERE

*Your answer:*

## Part 5 — First look at EEG: a seizure in CHB-MIT
CHB-MIT (Shoeb, 2009; PhysioNet) contains long-term scalp EEG of paediatric patients with epilepsy.
Record `chb01_03` (1 h, 23 bipolar channels, 256 Hz) contains a seizure from **2996 s to 3036 s**.
The cell below installs MNE (the standard Python EEG toolbox) and loads the course's small data helpers.

In [ ]:
!pip install -q mne

In [ ]:
"""Small loaders for the two open-access EEG seizure datasets used in the labs.

CHB-MIT  (Shoeb 2009, PhysioNet): paediatric scalp EEG, 23 bipolar channels, 256 Hz, 1 h records.
SeizeIT2 (Bhagubai et al. 2025, OpenNeuro ds005873): wearable behind-the-ear EEG, 2 channels, 256 Hz.
Files are downloaded once and cached in ./data (in Colab: lost when the runtime is recycled).
"""
import re
import urllib.request
from pathlib import Path

import numpy as np

DATA = Path("data")
CHBMIT = "https://physionet-open.s3.amazonaws.com/chbmit/1.0.0/"  # AWS mirror of PhysioNet, much faster
SEIZEIT2 = "https://s3.amazonaws.com/openneuro.org/ds005873/"
# the first 18 CHB-MIT channels form the longitudinal bipolar "double banana" montage
CHANNELS = ["FP1-F7", "F7-T7", "T7-P7", "P7-O1", "FP1-F3", "F3-C3", "C3-P3", "P3-O1", "FP2-F4",
            "F4-C4", "C4-P4", "P4-O2", "FP2-F8", "F8-T8", "T8-P8", "P8-O2", "FZ-CZ", "CZ-PZ"]


def fetch(url, name):
    """Download url to data/name once; return the local path."""
    out = DATA / name
    if not out.exists():
        out.parent.mkdir(parents=True, exist_ok=True)
        print("downloading", url)
        urllib.request.urlretrieve(url, out.with_suffix(".part"))
        out.with_suffix(".part").rename(out)  # no half-written files in the cache
    return out


def preprocess(raw, picks, fs):
    """Band-pass 0.5-40 Hz, resample to fs, return (channels, time) in microvolts."""
    raw.pick(picks).filter(0.5, 40, verbose="error").resample(fs, verbose="error")
    return raw.get_data() * 1e6


def windows(data, fs, win_s, events, around_s=None):
    """Cut (channels, time) into non-overlapping windows of win_s seconds.

    y = 1 if the window centre lies inside a seizure event (onset_s, end_s).
    around_s: keep only windows within around_s seconds of a seizure (None = keep all).
    """
    n = int(win_s * fs)
    k = data.shape[1] // n
    X = data[:, :k * n].reshape(data.shape[0], k, n).transpose(1, 0, 2)
    mid = (np.arange(k) + 0.5) * win_s
    y = np.zeros(k, dtype=np.int64)
    keep = np.zeros(k, dtype=bool) if around_s is not None else np.ones(k, dtype=bool)
    for a, b in events:
        y[(mid >= a) & (mid < b)] = 1
        if around_s is not None:
            keep |= (mid >= a - around_s) & (mid < b + around_s)
    return np.ascontiguousarray(X[keep], dtype=np.float32), y[keep]


def chbmit_seizures(patient="chb01"):
    """{'chb01_03': [(2996, 3036)], ...} parsed from the PhysioNet summary file."""
    text = fetch(f"{CHBMIT}{patient}/{patient}-summary.txt", f"chbmit/{patient}-summary.txt").read_text()
    out = {}
    for block in text.split("File Name: ")[1:]:
        t = [int(s) for s in re.findall(r"Seizure.*(?:Start|End) Time:\s*(\d+)", block)]
        out[block.split(".edf")[0]] = list(zip(t[::2], t[1::2]))
    return out


def load_chbmit(records=("chb01_03", "chb01_04", "chb01_15", "chb01_16", "chb01_18", "chb01_26"),
                win_s=4.0, fs=128):
    """CHB-MIT records -> X (n, 18, win_s*fs) in uV, y (n,) 1 = seizure, rec (n,) index into records."""
    import mne
    X, y, rec = [], [], []
    for i, r in enumerate(records):
        patient = r.split("_")[0]
        raw = mne.io.read_raw_edf(fetch(f"{CHBMIT}{patient}/{r}.edf", f"chbmit/{r}.edf"), preload=True, verbose="error")
        raw.rename_channels(lambda c: c.removesuffix("-0"))  # MNE renames the duplicated T8-P8 channel
        Xi, yi = windows(preprocess(raw, CHANNELS, fs), fs, win_s, chbmit_seizures(patient)[r])
        X.append(Xi), y.append(yi), rec.append(np.full(len(yi), i))
    return np.concatenate(X), np.concatenate(y), np.concatenate(rec)


def seizeit2_events(subject="sub-001"):
    """{run_path: [(onset_s, end_s), ...]} seizure events of every EEG run of one SeizeIT2 subject."""
    listing = urllib.request.urlopen(
        f"https://s3.amazonaws.com/openneuro.org?list-type=2&prefix=ds005873/{subject}/").read().decode()
    out = {}
    for stem in re.findall(r"<Key>ds005873/([^<]+?/eeg/[^<]+?)_events\.tsv</Key>", listing):
        rows = fetch(f"{SEIZEIT2}{stem}_events.tsv", f"seizeit2/{stem}_events.tsv").read_text().splitlines()[1:]
        cols = [r.split("\t") for r in rows]
        out[stem] = [(float(c[0]), float(c[0]) + float(c[1])) for c in cols if c[2].startswith("sz")]
    return out


def load_seizeit2(subject="sub-001", win_s=4.0, fs=128, around_s=1800):
    """Seizure-containing runs of one SeizeIT2 subject -> X (n, 2, win_s*fs) in uV, y (n,), run (n,).

    around_s keeps +-30 min around each seizure, otherwise one subject is >100 h of mostly background.
    """
    import mne
    X, y, run = [], [], []
    for i, (stem, events) in enumerate(e for e in seizeit2_events(subject).items() if e[1]):
        raw = mne.io.read_raw_edf(fetch(f"{SEIZEIT2}{stem}_eeg.edf", f"seizeit2/{stem}_eeg.edf"), preload=True, verbose="error")
        Xi, yi = windows(preprocess(raw, raw.ch_names[:2], fs), fs, win_s, events, around_s)
        X.append(Xi), y.append(yi), run.append(np.full(len(yi), i))
    return np.concatenate(X), np.concatenate(y), np.concatenate(run)


def band_power(X, fs=128):
    """Log band power (delta, theta, alpha, beta, low gamma) per channel: (n, ch, t) -> (n, ch*5)."""
    from scipy.signal import welch
    f, p = welch(X, fs=fs, nperseg=fs, axis=-1)
    bands = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 40)]
    bp = np.stack([p[..., (f >= a) & (f < b)].mean(-1) for a, b in bands], -1)
    return np.log(bp + 1e-10).reshape(len(X), -1).astype(np.float32)
import mne

raw = mne.io.read_raw_edf(fetch(CHBMIT + "chb01/chb01_03.edf", "chbmit/chb01_03.edf"), preload=True, verbose="error")
print(raw)
onset, offset = chbmit_seizures("chb01")["chb01_03"][0]
print("seizure:", onset, "-", offset, "s")

**Exercise 5.1.** Plot 8 channels from 30 s before to 30 s after seizure onset (vertically offset so they do not
overlap) and mark onset with a vertical line. Use `raw.get_data(picks=..., tmin=..., tmax=...)` (volts → ×1e6 for µV)
and `raw.info["sfreq"]`. What changes at the onset?

In [ ]:
picks = CHANNELS[:8]
pass  # YOUR CODE HERE

*Your answer:*

**Exercise 5.2 — teaser for the rest of the course.** The helper `load_chbmit()` cuts six records of patient chb01 into
4-s windows. Load it and answer: how many windows are there, what fraction is seizure, and what accuracy would a
"model" reach that always predicts *no seizure*? What does that tell you about accuracy as a metric?

In [ ]:
X, y, rec = load_chbmit()
print("X", X.shape, "y", y.shape, "records", np.unique(rec))
# TODO: compute seizure_fraction and the accuracy of always predicting 0 (trivial_acc)
pass  # YOUR CODE HERE
print(f"seizure windows: {y.sum()} ({100 * seizure_fraction:.2f} %), trivial accuracy: {100 * trivial_acc:.2f} %")

*Your answer:*

## Wrap-up
- A linear network is `nn.Linear`: $\hat{\mathbf y} = \mathbf W\mathbf x + \mathbf b$; MSE + gradient descent trains it.
- Always check hand-written gradients numerically; autograd does the bookkeeping for you.
- The training loop *forward → loss → zero_grad → backward → step* is the same for every model in this course.
- Learning rate and batch size interact; standardise inputs with **training** statistics only.